# LogicBench — Fractional Factorial Analysis (Question-Level Logistic Regression)

**Design**: $2^{4-1}_{IV}$, generator $T_2 = Q_1 Q_2 T_1$  
**Unit of analysis**: individual questions (binary correct/incorrect)  
**Model**: logistic regression — proper CIs and p-values at n ≈ 320

Factors:
- $Q_1$ = `use_shortcuts`
- $Q_2$ = `expand_query`  
- $T_1$ = `use_openie`  
- $T_2$ = `use_enrichment_kb`  (generated: $T_2 = Q_1 Q_2 T_1$)

In [ ]:
# Cell 1 — Imports
import json
import requests
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')
sns.set_palette('colorblind')

print('Libraries loaded.')

In [ ]:
# Cell 2 — Fractional factorial design matrix
# 2^{4-1}_IV with generator T2 = Q1 * Q2 * T1
# Coded ±1: +1 = factor ON, -1 = factor OFF

DESIGN = pd.DataFrame({
    'run': [1,  2,  3,  4,  5,  6,  7,  8],
    'Q1':  [-1, +1, -1, +1, -1, +1, -1, +1],   # use_shortcuts
    'Q2':  [-1, -1, +1, +1, -1, -1, +1, +1],   # expand_query
    'T1':  [-1, -1, -1, -1, +1, +1, +1, +1],   # use_openie
})
DESIGN['T2'] = DESIGN['Q1'] * DESIGN['Q2'] * DESIGN['T1']  # generated factor

# Verify generator holds
assert (DESIGN['T2'] == DESIGN['Q1'] * DESIGN['Q2'] * DESIGN['T1']).all()

print('Fractional factorial design (2^{4-1}_IV):')
print('Generator: T2 = Q1 * Q2 * T1')
print()
print(DESIGN.to_string(index=False))
print()
print('Aliasing structure (Resolution IV):')
print('  ME(Q1) aliased with 3FI(Q2,T1,T2)')
print('  ME(Q2) aliased with 3FI(Q1,T1,T2)')
print('  ME(T1) aliased with 3FI(Q1,Q2,T2)')
print('  ME(T2) aliased with 3FI(Q1,Q2,T1)')
print('  2FI(Q1,Q2) aliased with 2FI(T1,T2)')
print('  2FI(Q1,T1) aliased with 2FI(Q2,T2)')
print('  2FI(Q2,T1) aliased with 2FI(Q1,T2)')

In [ ]:
# Cell 3 — File-to-run mapping
# Note: run4 has two files; we use the later re-run (053133)

BASE_URL = (
    'https://raw.githubusercontent.com/pgallardo/article_logic_ai_2026'
    '/main/experiments/logicBench/results/'
)

RUN_FILES = {
    1: '20260401_005326_propositional_logic_modus_tollens_config_run1_openAI.json',
    2: '20260401_052127_propositional_logic_modus_tollens_config_run2_openAI.json',
    3: '20260401_052819_propositional_logic_modus_tollens_config_run3_openAI.json',
    4: '20260401_053133_propositional_logic_modus_tollens_config_run4_openAI.json',  # re-run
    5: '20260401_052404_propositional_logic_modus_tollens_config_run5_openAI.json',
    6: '20260401_015324_propositional_logic_modus_tollens_config_run6_openAI.json',
    7: '20260401_005455_propositional_logic_modus_tollens_config_run7_openAI.json',
    8: '20260401_000948_propositional_logic_modus_tollens_config_run8_openAI.json',  # incomplete (28/40)
}

In [ ]:
# Cell 4 — Load question-level data from all runs

def load_run(run_num, filename, base_url=BASE_URL):
    """
    Fetch one result JSON and return a DataFrame of question-level outcomes.
    Each row = one question with its design factors and binary correct label.
    """
    resp = requests.get(base_url + filename)
    resp.raise_for_status()
    data = resp.json()

    design_row = DESIGN[DESIGN['run'] == run_num].iloc[0]

    rows = []
    for entry in data.get('results', []):
        rows.append({
            'run':          run_num,
            'Q1':           design_row['Q1'],
            'Q2':           design_row['Q2'],
            'T1':           design_row['T1'],
            'T2':           design_row['T2'],
            'correct':      int(bool(entry.get('is_correct', False))),
            'ground_truth': str(entry.get('ground_truth', '')).strip().lower(),
            'sample_id':    entry.get('sample_id'),
            'question_idx': entry.get('question_idx'),
        })
    return pd.DataFrame(rows)


frames = []
for run_num, fname in RUN_FILES.items():
    df_run = load_run(run_num, fname)
    acc = df_run['correct'].mean()
    n   = len(df_run)
    print(f'Run {run_num}: {n:3d} questions | accuracy = {acc:.3f}'
          + (' ⚠ incomplete' if n < 40 else ''))
    frames.append(df_run)

df = pd.concat(frames, ignore_index=True)
print(f'\nTotal: {len(df)} questions across {df["run"].nunique()} runs')

In [ ]:
# Cell 5 — Sanity checks

# 1. Generator: T2 must equal Q1*Q2*T1 in every row
assert (df['T2'] == df['Q1'] * df['Q2'] * df['T1']).all(), 'Generator violated!'
print('✓ T2 = Q1 × Q2 × T1 holds for every question')

# 2. Orthogonality check: pairwise dot products of factor columns should be 0
# (up to weighting by unequal n across runs)
for fa, fb in [('Q1','Q2'), ('Q1','T1'), ('Q1','T2'), ('Q2','T1'), ('Q2','T2'), ('T1','T2')]:
    dot = (df[fa] * df[fb]).sum()
    print(f'  dot({fa},{fb}) = {dot:+4d}  '
          + ('✓ orthogonal' if dot == 0 else '⚠ non-zero (run 8 is shorter)'))

# 3. Label distribution
print('\nLabel distribution per run:')
pivot = df.groupby(['run', 'ground_truth'])['correct'].count().unstack(fill_value=0)
print(pivot)

# 4. Overall accuracy by label
print('\nOverall accuracy by ground-truth label:')
print(df.groupby('ground_truth')['correct'].agg(['mean', 'count']).rename(
    columns={'mean': 'accuracy', 'count': 'n'}).round(3))

## Logistic Regression Models

**Why logistic regression?**  
Each question is a binary outcome. The 320 questions give us proper degrees of freedom to estimate all main effects and aliased two-factor interactions simultaneously — something that isn't possible with only 8 run-level averages.

**Two caveats:**  
1. Questions within the same run are not fully independent (same pipeline config). Standard errors are slightly anti-conservative; we report them as-is since the 40 questions per run differ in document content.  
2. Run 8 contributes only 28 questions (incomplete). It is included as-is; its reduced weight is handled naturally by the regression.

**Aliasing reminder:**  
The coefficient for $T_2$ estimates $\text{ME}(T_2) + \text{3FI}(Q_1 Q_2 T_1)$ — indistinguishable in this design. Same applies to the 2FI pairs below.

In [ ]:
# Cell 6 — Model A: main effects only
# logit(correct) = β0 + β_Q1·Q1 + β_Q2·Q2 + β_T1·T1 + β_T2·T2
# (T2 is linearly independent of Q1,Q2,T1 in ±1 coding — no rank deficiency)

X_main = sm.add_constant(df[['Q1', 'Q2', 'T1', 'T2']])
model_main = sm.Logit(df['correct'], X_main).fit(disp=False)

print('Model A — Main effects only')
print('='*60)
print(model_main.summary2().tables[1].to_string())

In [ ]:
# Cell 7 — Model B: main effects + aliased two-factor interactions
# Estimable 2FI pairs (one representative per aliased pair):
#   Q1×Q2  (= T1×T2)
#   Q1×T1  (= Q2×T2)
#   Q2×T1  (= Q1×T2)
# This model saturates the 8 design points (intercept + 7 terms = 8 params).

df['Q1xQ2'] = df['Q1'] * df['Q2']
df['Q1xT1'] = df['Q1'] * df['T1']
df['Q2xT1'] = df['Q2'] * df['T1']

X_full = sm.add_constant(df[['Q1', 'Q2', 'T1', 'T2', 'Q1xQ2', 'Q1xT1', 'Q2xT1']])
model_full = sm.Logit(df['correct'], X_full).fit(disp=False)

print('Model B — Main effects + aliased 2FIs (saturated at run level)')
print('='*65)
print(model_full.summary2().tables[1].to_string())

# Likelihood-ratio test: do 2FIs improve over main-effects model?
lr_stat = 2 * (model_full.llf - model_main.llf)
lr_df   = model_full.df_model - model_main.df_model
from scipy.stats import chi2
lr_pval = chi2.sf(lr_stat, lr_df)
print(f'\nLR test (2FIs vs main only): χ²({int(lr_df)}) = {lr_stat:.3f}, p = {lr_pval:.4f}')

In [ ]:
# Cell 8 — Label-stratified models (yes / no separately)
# Tests whether the factor effects differ between entailment and non-entailment questions

label_models = {}
for label in ['yes', 'no']:
    sub = df[df['ground_truth'] == label]
    X   = sm.add_constant(sub[['Q1', 'Q2', 'T1', 'T2']])    
    m = sm.Logit(sub['correct'], X).fit(disp=False, maxiter=200)
    if not m.converged:
        print(f'  ⚠ WARNING: model did not converge for label={label}')
    label_models[label] = m
    
    label_models[label] = m
    print(f'\n--- Label: {label.upper()}  (n={len(sub)}, base accuracy={sub["correct"].mean():.3f}) ---')
    print(m.summary2().tables[1].to_string())

In [ ]:
# Cell 9 — Accuracy-scale effects (direct contrasts)
# Complement to log-odds: raw accuracy difference when a factor goes from − to +
# Useful for reporting alongside the logistic regression

def accuracy_effects(data, factors=('Q1','Q2','T1','T2')):
    rows = []
    for f in factors:
        hi  = data[data[f] == +1]['correct'].mean()
        lo  = data[data[f] == -1]['correct'].mean()
        rows.append({'factor': f, 'effect': hi - lo,
                     'acc_high': hi, 'acc_low': lo})
    return pd.DataFrame(rows).set_index('factor')

print('Accuracy-scale main effects (Δ accuracy when factor switches − → +)\n')
for name, subset in [('All',      df),
                     ('Yes only', df[df['ground_truth']=='yes']),
                     ('No only',  df[df['ground_truth']=='no'])]:
    eff = accuracy_effects(subset)
    print(f'{name} (n={len(subset)}):')
    print(eff.round(3).to_string())
    print()

In [ ]:
# Cell 10 — Helper: extract coefficients, CIs, p-values from a fitted model

FACTOR_LABELS = {
    'Q1':    r'$Q_1$: Shortcuts',
    'Q2':    r'$Q_2$: Expand',
    'T1':    r'$T_1$: OpenIE',
    'T2':    r'$T_2$: Enrich',
    'Q1xQ2': r'$Q_1 \times Q_2$ (=$T_1 T_2$)',
    'Q1xT1': r'$Q_1 \times T_1$ (=$Q_2 T_2$)',
    'Q2xT1': r'$Q_2 \times T_1$ (=$Q_1 T_2$)',
}

def model_to_df(model):
    ci   = model.conf_int()
    rows = []
    for var in model.params.index:
        if var == 'const':
            continue
        rows.append({
            'var':     var,
            'label':   FACTOR_LABELS.get(var, var),
            'coef':    model.params[var],
            'ci_lo':   ci.loc[var, 0],
            'ci_hi':   ci.loc[var, 1],
            'pval':    model.pvalues[var],
            'sig':     model.pvalues[var] < 0.05,
        })
    return pd.DataFrame(rows)

In [ ]:
# Cell 11 — Figure 1: Forest plot — main effects log-odds with 95% CIs
# Three panels: all questions / yes-only / no-only

MAIN_VARS = ['Q1', 'Q2', 'T1', 'T2']

datasets = [
    ('All questions',       model_main),
    ('Yes (entailment)',    label_models['yes']),
    ('No (not entailment)', label_models['no']),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

for ax, (title, model) in zip(axes, datasets):
    eff = model_to_df(model)
    eff = eff[eff['var'].isin(MAIN_VARS)].reset_index(drop=True)

    ypos   = np.arange(len(eff))
    colors = ['#2166ac' if s else '#b2b2b2' for s in eff['sig']]
    xerr   = np.array([eff['coef'] - eff['ci_lo'],
                       eff['ci_hi'] - eff['coef']])

    ax.barh(ypos, eff['coef'], xerr=xerr, color=colors,
            edgecolor='black', linewidth=0.5, capsize=4, height=0.55)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_yticks(ypos)
    ax.set_yticklabels(eff['label'], fontsize=10)
    ax.set_xlabel('Log-odds coefficient', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')

    for i, row in eff.iterrows():
        marker = '*' if row['sig'] else ''
        ax.text(row['ci_hi'] + 0.03, i,
                f"p={row['pval']:.3f}{marker}",
                va='center', fontsize=8)

blue_patch = mpatches.Patch(color='#2166ac', label='p < 0.05')
grey_patch = mpatches.Patch(color='#b2b2b2', label='p ≥ 0.05')
axes[2].legend(handles=[blue_patch, grey_patch], loc='lower right', fontsize=9)

plt.suptitle('LogicBench: Main Effects (log-odds, 95% CI)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('logicbench_forest_main.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 12 — Figure 2: Accuracy-scale effects by label
# Direct contrast: mean(correct | factor=+) − mean(correct | factor=−)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
label_names = {'yes': 'Yes — entailment', 'no': 'No — not entailment'}
factor_tick = [r'$Q_1$\nShortcuts', r'$Q_2$\nExpand',
               r'$T_1$\nOpenIE',    r'$T_2$\nEnrich']

for ax, label in zip(axes, ['yes', 'no']):
    eff = accuracy_effects(df[df['ground_truth'] == label])
    colors = ['#27ae60' if e > 0 else '#e74c3c' for e in eff['effect']]
    bars = ax.bar(range(4), eff['effect'], color=colors,
                  edgecolor='black', alpha=0.85, width=0.55)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-0.45, 0.45)
    ax.set_xticks(range(4))
    ax.set_xticklabels([r'$Q_1$' + '\nShortcuts', r'$Q_2$' + '\nExpand',
                        r'$T_1$' + '\nOpenIE', r'$T_2$' + '\nEnrich'], fontsize=10)
    ax.set_ylabel('Δ Accuracy')
    ax.set_title(label_names[label], fontsize=11, fontweight='bold')

    for bar, e in zip(bars, eff['effect']):
        ax.text(bar.get_x() + bar.get_width() / 2,
                e + (0.015 if e >= 0 else -0.025),
                f'{e:+.3f}', ha='center',
                va='bottom' if e >= 0 else 'top', fontsize=9)

plt.suptitle('LogicBench: Accuracy-scale Main Effects by Label', fontsize=13)
plt.tight_layout()
plt.savefig('logicbench_accuracy_effects.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 13 — Figure 3: Full model (main + 2FIs) forest plot

eff_full_df = model_to_df(model_full)
ypos   = np.arange(len(eff_full_df))
colors = ['#2166ac' if s else '#b2b2b2' for s in eff_full_df['sig']]
xerr   = np.array([eff_full_df['coef'] - eff_full_df['ci_lo'],
                   eff_full_df['ci_hi'] - eff_full_df['coef']])

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(ypos, eff_full_df['coef'], xerr=xerr, color=colors,
        edgecolor='black', linewidth=0.5, capsize=4, height=0.55)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(ypos)
ax.set_yticklabels(eff_full_df['label'], fontsize=10)
ax.set_xlabel('Log-odds coefficient', fontsize=10)
ax.set_title('LogicBench: Model B — Main Effects + Aliased 2FIs (log-odds, 95% CI)',
             fontsize=12, fontweight='bold')

# Separator line between MEs and 2FIs
ax.axhline(3.5, color='gray', linewidth=0.8, linestyle='--')
ax.text(ax.get_xlim()[0], 3.6, 'Main effects', fontsize=8, color='gray')
ax.text(ax.get_xlim()[0], 4.1, '2FIs (aliased pairs)', fontsize=8, color='gray')

for i, row in eff_full_df.iterrows():
    marker = '*' if row['sig'] else ''
    ax.text(row['ci_hi'] + 0.03, i,
            f"p={row['pval']:.3f}{marker}",
            va='center', fontsize=8)

ax.legend(handles=[mpatches.Patch(color='#2166ac', label='p < 0.05'),
                   mpatches.Patch(color='#b2b2b2', label='p ≥ 0.05')],
          loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('logicbench_forest_full.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 14 — Summary table (console)

print('=' * 70)
print(f'LOGICBENCH — MAIN EFFECTS SUMMARY (logistic regression, n={len(df)})')
print('=' * 70)
print(f'Overall accuracy : {df["correct"].mean():.3f}')
print(f'  Yes (entailment)    : {df[df["ground_truth"]=="yes"]["correct"].mean():.3f}')
print(f'  No (not entailment) : {df[df["ground_truth"]=="no"]["correct"].mean():.3f}')
print()

eff_all = model_to_df(model_main)
eff_yes = model_to_df(label_models['yes'])
eff_no  = model_to_df(label_models['no'])

print(f'{"Factor":<22} {"All":>10}  {"Yes only":>12}  {"No only":>12}')
print(f'{"":<22} {"coef (p)":>10}  {"coef (p)":>12}  {"coef (p)":>12}')
print('-' * 62)
for var in MAIN_VARS:
    ra = eff_all[eff_all['var']==var].iloc[0]
    ry = eff_yes[eff_yes['var']==var].iloc[0]
    rn = eff_no[eff_no['var']==var].iloc[0]

    def fmt(r):
        s = '*' if r['sig'] else ' '
        return f"{r['coef']:+.3f}{s}({r['pval']:.3f})"

    print(f"{FACTOR_LABELS[var]:<22} {fmt(ra):>10}  {fmt(ry):>12}  {fmt(rn):>12}")

print()
print('* p < 0.05   Coefficients are log-odds ratios.')
print('Note: T2 coefficient estimates ME(T2) + 3FI(Q1,Q2,T1) — aliased.')

In [ ]:
# Cell 15 — LaTeX table output

print('% LaTeX: LogicBench Main Effects (logistic regression)')
print(r'\begin{table}[t]')
print(r'\centering')
print(r'\caption{LogicBench main effects: logistic regression coefficients'
      r' (log-odds, 95\% CI). Asterisk denotes $p<0.05$.'
      r' $T_2$ coefficient estimates $\text{ME}(T_2) + \text{3FI}(Q_1 Q_2 T_1)$.}')
print(r'\label{tab:logicbench_logit}')
print(r'\begin{tabular}{l ccc}')
print(r'\toprule')
print(r'Factor & All & Yes (entailment) & No (not entailment) \\')
print(r'\midrule')

factor_tex = {
    'Q1': r'$Q_1$: Shortcuts',
    'Q2': r'$Q_2$: Expand',
    'T1': r'$T_1$: OpenIE',
    'T2': r'$T_2$: Enrich$^\dagger$',
}

for var in MAIN_VARS:
    ra = eff_all[eff_all['var']==var].iloc[0]
    ry = eff_yes[eff_yes['var']==var].iloc[0]
    rn = eff_no[eff_no['var']==var].iloc[0]

    def tex_cell(r):
        c = f"{r['coef']:+.3f}"
        ci = f"[{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}]"
        sig = r'^*' if r['sig'] else ''
        return f"${c}{sig}$ {ci}"

    print(f"{factor_tex[var]} & {tex_cell(ra)} & {tex_cell(ry)} & {tex_cell(rn)} \\\\")

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')